# Small Cap Cleaning and Metrics

Run `small_cap_excel_to_csv.ipynb` first. This notebook reads the five CSVs in `Small_cap/clean`, cleans each 2022–2025 NAV series, adds daily returns, running maximum and drawdown, and saves per-fund analysis CSVs and the fund-level metrics table under `Small_cap/analysis`.


In [18]:
import pandas as pd
import numpy as np
from pathlib import Path


In [19]:
funds = {
    "Nippon India Small Cap": "Small_cap/clean/nippon_small_cap.csv",
    "SBI Small Cap": "Small_cap/clean/sbi_small_cap.csv",
    "HDFC Small Cap": "Small_cap/clean/hdfc_small_cap.csv",
    "Axis Small Cap": "Small_cap/clean/axis_small_cap.csv",
    "Kotak Small Cap": "Small_cap/clean/kotak_small_cap.csv"
}


In [20]:
def clean_nav_data(file_path):
    df = pd.read_csv(file_path)
    df.columns = df.columns.astype(str).str.strip()
    df = df[["Date", "Fund", "Category", "NAV"]]

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["NAV"] = pd.to_numeric(df["NAV"], errors="coerce")
    df["Fund"] = df["Fund"].astype(str).str.strip()
    df["Category"] = df["Category"].astype(str).str.strip()

    df = df[
        (df["Date"] >= "2022-01-01") &
        (df["Date"] <= "2025-12-31")
    ]
    df = df.dropna(subset=["Date", "NAV"])
    df = df.sort_values("Date").reset_index(drop=True)

    if df.empty:
        raise ValueError(f"No usable NAV rows found in {file_path}")

    df["Daily_Return"] = df["NAV"].pct_change()
    df["Running_Max"] = df["NAV"].cummax()
    df["Drawdown"] = (df["NAV"] - df["Running_Max"]) / df["Running_Max"]
    return df


In [21]:
def calculate_cagr(df):
    start_nav = df["NAV"].iloc[0]
    end_nav = df["NAV"].iloc[-1]
    years = (df["Date"].iloc[-1] - df["Date"].iloc[0]).days / 365.25
    return (end_nav / start_nav) ** (1 / years) - 1


def calculate_volatility(df):
    return df["Daily_Return"].dropna().std() * np.sqrt(252)


def calculate_max_drawdown(df):
    return df["Drawdown"].min()


RISK_FREE_RATE = 0.06
TRADING_DAYS = 252

def calculate_sharpe(df, risk_free_rate=RISK_FREE_RATE):
    daily_returns = df["Daily_Return"].dropna()
    if len(daily_returns) < 2:
        return np.nan
    daily_rf = (1 + risk_free_rate) ** (1 / TRADING_DAYS) - 1
    daily_std = daily_returns.std()
    if pd.isna(daily_std) or daily_std == 0:
        return np.nan
    excess_returns = daily_returns - daily_rf
    return (excess_returns.mean() / daily_std) * np.sqrt(TRADING_DAYS)


In [22]:
cleaned_data = {}
for fund_name, file_path in funds.items():
    cleaned_data[fund_name] = clean_nav_data(file_path)

analysis_dir = Path("Small_cap/analysis")
analysis_dir.mkdir(parents=True, exist_ok=True)

results = []
for fund_name, df in cleaned_data.items():
    filename = fund_name.lower().replace(" ", "_").replace("india_", "") + "_analysis.csv"
    analysis_path = analysis_dir / filename
    df.to_csv(analysis_path, index=False)
    print(f"Saved analysis data: {analysis_path}")

    results.append({
        "Fund": fund_name,
        "Start Date": df["Date"].min(),
        "End Date": df["Date"].max(),
        "Observations": len(df),
        "Start NAV": df["NAV"].iloc[0],
        "End NAV": df["NAV"].iloc[-1],
        "CAGR": calculate_cagr(df),
        "Volatility": calculate_volatility(df),
        "Maximum Drawdown": calculate_max_drawdown(df),
        "Sharpe Ratio": calculate_sharpe(df)
    })

small_cap_results = pd.DataFrame(results)
display(small_cap_results)


Saved analysis data: Small_cap/analysis/nippon_small_cap_analysis.csv
Saved analysis data: Small_cap/analysis/sbi_small_cap_analysis.csv
Saved analysis data: Small_cap/analysis/hdfc_small_cap_analysis.csv
Saved analysis data: Small_cap/analysis/axis_small_cap_analysis.csv
Saved analysis data: Small_cap/analysis/kotak_small_cap_analysis.csv


,Fund,Start Date,End Date,Observations,Start NAV,End NAV,CAGR,Volatility,Maximum Drawdown,Sharpe Ratio
0,Nippon India Small Cap,2022-01-03,2025-12-31,980,96.4493,187.0214,0.180446,0.162468,-0.242087,0.772565
1,SBI Small Cap,2022-01-03,2025-12-31,986,117.8295,193.7724,0.132715,0.133037,-0.224246,0.585580
2,HDFC Small Cap,2022-01-03,2025-12-31,980,85.0160,157.8940,0.167763,0.158607,-0.226141,0.717251
3,Axis Small Cap,2022-01-03,2025-12-31,985,69.6600,121.6300,0.149844,0.137202,-0.194265,0.684724
4,Kotak Small Cap,2022-01-03,2025-12-31,980,189.9220,294.5410,0.116196,0.138081,-0.246729,0.465522


In [23]:
results_path = Path("Small_cap/analysis/small_cap_fund_results.csv")
small_cap_results.to_csv(results_path, index=False)
print(f"Fund-level results saved to: {results_path}")


Fund-level results saved to: Small_cap/analysis/small_cap_fund_results.csv
